# Fluid-Structure Interaction · Fixed-block flow

[Full course sequence](../../ai4sci/README.md) · [Start notebook](../../Start_Here.ipynb)

All levels are retained and use PhysicsNeMo **2.2.2** `FullyConnected`, SymPy `PDE`, `PhysicsInformer`, and explicit PyTorch training loops. Complete `student_equations` in the linked `.py` file. Until that function is completed, the default run stops with an explanatory error. If you get stuck, set `USE_REFERENCE=True` to run and compare the completed implementation.

**Learning workflow:** inspect the equations and conditions → edit and save the linked `.py` file → run it → inspect the PDE and condition errors and the predictions. A successful short run does not establish convergence. Each run writes to a new directory.

`PhysicsInformer` computes spatial derivatives from `coordinates`. For the current API, the training code computes time derivatives with PyTorch autograd and supplies keys such as `u__t` and `u__t__t`. Read `loss_terms` and the explicit `optimizer.zero_grad → backward → step` loop in each training file.

Instructor automation can execute these same cells with the environment variables `AI4SCI_REFERENCE=1`, `AI4SCI_DEVICE=cpu`, and `AI4SCI_STEPS=2`. The default remains the student exercise mode.


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import os
import subprocess
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "challenge" / "fuild" / "chip_2d_l1.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open the notebook from inside the repository.")
LAB_DIR = ROOT / "challenge" / "fuild"
USE_REFERENCE = os.environ.get("AI4SCI_REFERENCE", "0").lower() in {"1", "true", "yes"}  # Default: student exercise mode.
DEVICE = os.environ.get("AI4SCI_DEVICE", "auto")
STEPS = int(os.environ.get("AI4SCI_STEPS", "200"))  # Verify runtime and accuracy on the event GPU.
SEED = 42
RUN_TAG = datetime.now().strftime("%Y%m%d-%H%M%S-%f")


## Level 1 · 2D flow

![Original channel and single block](images/chip_2d.png)

Subtract the block $[-1,0]\times[-.5,.1]$ from the channel $[-2.5,2.5]\times[-.5,.5]$. This is a steady incompressible Navier–Stokes problem with $\nu=.02,\rho=1$.
$$u_x+v_y=0,\quad uu_x+vu_y+p_x/\rho-\nu\Delta u=0,\quad uv_x+vv_y+p_y/\rho-\nu\Delta v=0.$$
The inlet has $u=1.5(1-4y^2),v=0$, the outlet has $p=0$, and the walls and block have no-slip conditions. The target flow rate through each vertical fluid cross-section is 1. `fluid_geometry.py` samples the interior outside the block and the exposed walls. Inlet and outlet points are excluded from no-slip samples.
The bundled OpenFOAM data provide a separate numerical comparison, not an analytic solution. This code models flow around a fixed obstacle; it does not solve structural deformation.

### Code and exercise

Open [chip_2d_l1.py](chip_2d_l1.py) and inspect `reference_equations`, `student_equations`, `loss_terms`, and `main`. Write the required dictionary of PDE residuals in `student_equations`, then save with **Ctrl+S / ⌘S**. Identify where the applicable initial, boundary, and integral conditions enter the loss.

Start with a small run equivalent to `--steps 2 --device cpu --reference` to check the complete input/output path. The student and reference implementations share the same sampling and evaluation code.

### SDF weighting and integrated flow rate

The original SDF weighting is retained. Inside the fluid domain, compute the minimum distance $d>0$ to walls and blocks, then multiply the squared continuity and momentum residuals by $2d$. This training choice reduces weights near walls and corners; it does not mean that the physical errors there are small. Also inspect `*_unweighted_rmse` in `metrics.json`. The SDF image linked in the original material is absent from the repository, so inspect the actual `sdf_weight` function.

For each vertical cross-section, compute $Q(x,t)=\int_{\text{fluid}}u(x,y,t)\,dy$ and match the inlet flow rate $\int_{-.5}^{.5}1.5(1-4y^2)\,dy=1$. A cross-section that intersects a block is integrated only over its fluid segment. `flux_lines` is the number of cross-sections, and `flux_points` is the number of midpoint quadrature points per cross-section. Compare the roles of the pointwise continuity equation and this integral condition.


In [ ]:
RUN_COMPLETED = False
result_dir = LAB_DIR / "outputs" / f"chip_2d_l1-{RUN_TAG}"
command = [sys.executable, str(LAB_DIR / "chip_2d_l1.py"),
           "--steps", str(STEPS), "--seed", str(SEED), "--device", DEVICE,
           "--output-dir", str(result_dir)]
if USE_REFERENCE:
    command.append("--reference")
subprocess.run(command, cwd=LAB_DIR, check=True)
RUN_COMPLETED = True


### Inspect the actual results

The first and last held-out rows in `loss.csv` are evaluated at **identical points**. Intermediate rows describe freshly sampled training minibatches. Distinguish the initial/final errors from the PDE and condition residuals in `metrics.json`. A reference error is included only when a comparable reference is available. `model.pt` stores the model state and configuration; `predictions.npz` contains the actual prediction arrays.


In [ ]:
if not RUN_COMPLETED:
    raise RuntimeError("The current training run has not completed.")
metrics = json.loads((result_dir / "metrics.json").read_text())
assert metrics["steps"] == STEPS and metrics["seed"] == SEED
print(json.dumps(metrics, indent=2))
preview = result_dir / "preview.png"
if preview.is_file():
    from IPython.display import display, Image
    display(Image(filename=str(preview)))
else:
    print("Plotting dependencies are unavailable. Inspect the actual arrays in predictions.npz.")


## Level 2 · Multiple Blocks in Channel

Use the same channel, fluid properties, and inlet/outlet conditions with three blocks.

| Block | x range | y range |
|---|---|---|
| 1 | $[-1,-.4]$ | $[-.5,-.1]$ |
| 2 | $[.2,.7]$ | $[-.5,0]$ |
| 3 | $[1.2,1.6]$ | $[-.5,-.15]$ |

Exclude the block interiors when integrating each cross-section. Inspect the fluid height and midpoint quadrature in `sample_flux`. No independent CFD reference is provided for this level. Interpret the physics residuals and condition errors separately.

### Code and exercise

Open [chip_2d_l2.py](chip_2d_l2.py) and inspect `reference_equations`, `student_equations`, `loss_terms`, and `main`. Write the required dictionary of PDE residuals in `student_equations`, then save with **Ctrl+S / ⌘S**. Identify where the applicable initial, boundary, and integral conditions enter the loss.

Start with a small run equivalent to `--steps 2 --device cpu --reference` to check the complete input/output path. The student and reference implementations share the same sampling and evaluation code.

The physical interpretation covers wakes behind multiple blocks, flow interference between blocks, and recirculation. Inspect velocity vectors or regions with $u<0$ for evidence of recirculation. A color map alone does not verify a vortex.


In [ ]:
RUN_COMPLETED = False
result_dir = LAB_DIR / "outputs" / f"chip_2d_l2-{RUN_TAG}"
command = [sys.executable, str(LAB_DIR / "chip_2d_l2.py"),
           "--steps", str(STEPS), "--seed", str(SEED), "--device", DEVICE,
           "--output-dir", str(result_dir)]
if USE_REFERENCE:
    command.append("--reference")
subprocess.run(command, cwd=LAB_DIR, check=True)
RUN_COMPLETED = True


### Inspect the actual results

The first and last held-out rows in `loss.csv` are evaluated at **identical points**. Intermediate rows describe freshly sampled training minibatches. Distinguish the initial/final errors from the PDE and condition residuals in `metrics.json`. A reference error is included only when a comparable reference is available. `model.pt` stores the model state and configuration; `predictions.npz` contains the actual prediction arrays.


In [ ]:
if not RUN_COMPLETED:
    raise RuntimeError("The current training run has not completed.")
metrics = json.loads((result_dir / "metrics.json").read_text())
assert metrics["steps"] == STEPS and metrics["seed"] == SEED
print(json.dumps(metrics, indent=2))
preview = result_dir / "preview.png"
if preview.is_file():
    from IPython.display import display, Image
    display(Image(filename=str(preview)))
else:
    print("Plotting dependencies are unavailable. Inspect the actual arrays in predictions.npz.")


## Level 3 · Time-Dependent Flow

Retain the single block, density, and boundary conditions from Level 1, and add the time interval $[0,10]$. As in the original Level 3, reduce the viscosity to $\nu=.01$ (Levels 1 and 2 use $.02$).
$$u_t+uu_x+vu_y+p_x/\rho-\nu\Delta u=0,\quad v_t+uv_x+vv_y+p_y/\rho-\nu\Delta v=0.$$
The interior values of $u,v,p$ are zero at $t=0$, while the original inlet velocity is applied at positive times. This impulsive-start setup is not smoothly compatible at the inlet at the initial time, so inspect errors near that region. Flux integration uses **the same time** at every point of each cross-section.
The inputs are $(x,y,t)$ and the outputs are $(u,v,p)$. This run alone does not provide an independent unsteady CFD reference or verify vortex-shedding frequency.

### Code and exercise

Open [chip_2d_l3.py](chip_2d_l3.py) and inspect `reference_equations`, `student_equations`, `loss_terms`, and `main`. Write the required dictionary of PDE residuals in `student_equations`, then save with **Ctrl+S / ⌘S**. Identify where the applicable initial, boundary, and integral conditions enter the loss.

Start with a small run equivalent to `--steps 2 --device cpu --reference` to check the complete input/output path. The student and reference implementations share the same sampling and evaluation code.

### Temporal evolution and vortex interpretation

The original observation topics are retained: startup flow, time-dependent wakes, vorticity $\omega=v_x-u_y$, and possible vortex shedding. A single snapshot cannot establish periodicity. Inspect fields at multiple times and velocity/pressure time series at points in the wake. If sufficient repeated variation is observed, define a frequency $f$ and $St=fL/U$. Do not use the representative Strouhal number of a circular cylinder as a reference answer for this block attached to the channel floor. Choosing $U=1.5,L=.6$ gives $Re=UL/\nu=90$; always state the characteristic length and velocity used. The current output is a mid-time cross-section. Verifying a shedding period requires additional time-series inference and independent CFD comparison.


In [ ]:
RUN_COMPLETED = False
result_dir = LAB_DIR / "outputs" / f"chip_2d_l3-{RUN_TAG}"
command = [sys.executable, str(LAB_DIR / "chip_2d_l3.py"),
           "--steps", str(STEPS), "--seed", str(SEED), "--device", DEVICE,
           "--output-dir", str(result_dir)]
if USE_REFERENCE:
    command.append("--reference")
subprocess.run(command, cwd=LAB_DIR, check=True)
RUN_COMPLETED = True


### Inspect the actual results

The first and last held-out rows in `loss.csv` are evaluated at **identical points**. Intermediate rows describe freshly sampled training minibatches. Distinguish the initial/final errors from the PDE and condition residuals in `metrics.json`. A reference error is included only when a comparable reference is available. `model.pt` stores the model state and configuration; `predictions.npz` contains the actual prediction arrays.


In [ ]:
if not RUN_COMPLETED:
    raise RuntimeError("The current training run has not completed.")
metrics = json.loads((result_dir / "metrics.json").read_text())
assert metrics["steps"] == STEPS and metrics["seed"] == SEED
print(json.dumps(metrics, indent=2))
preview = result_dir / "preview.png"
if preview.is_file():
    from IPython.display import display, Image
    display(Image(filename=str(preview)))
else:
    print("Plotting dependencies are unavailable. Inspect the actual arrays in predictions.npz.")


## Check your understanding and continue

- How do the inputs, outputs, equations, and initial/boundary conditions change between levels?
- Are training minibatch loss and error at fixed validation points the same metric?
- Does the problem have an independent reference? If so, do its assumptions match the current configuration?
- Compare changes to sample counts, training steps, and condition weights using new run directories and saved configurations.

[Next challenge](../climate/Multi-Physics_Climate_Modeling.ipynb) · [Full course sequence](../../ai4sci/README.md)

Adapted from the original OpenHackathons materials, with the file-specific copyright notices retained. [License](../../LICENSE).


--- 

Don't forget to check out additional [Open Hackathons Resources](https://www.openhackathons.org/s/technical-resources) and join our [OpenACC and Hackathons Slack Channel](https://www.openacc.org/community#slack) to share your experience and get more help from the community.

---

# Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials may include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.